In [2]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_squared_log_error
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    xgb_available = True
except:
    xgb_available = False

try:
    from lightgbm import LGBMRegressor
    lgbm_available = True
except:
    lgbm_available = False

try:
    from catboost import CatBoostRegressor
    cat_available = True
except:
    cat_available = False

# -----------------------------
# MLflow Setup
# -----------------------------
import mlflow
import mlflow.sklearn

mlflow.set_experiment("Bike Rental Demand Prediction")

def save_training_data(df, filename="training_data.csv"):
    os.makedirs("artifacts", exist_ok=True)
    path = os.path.join("artifacts", filename)
    df.to_csv(path, index=False)
    return path

# -----------------------------
# Load + Feature Engineering
# -----------------------------
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

def add_datetime_features(df):
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['hour'] = df['datetime'].dt.hour
    df['weekday'] = df['datetime'].dt.weekday
    df['month'] = df['datetime'].dt.month
    df['year'] = df['datetime'].dt.year
    return df

train = add_datetime_features(train)
test = add_datetime_features(test)

features = [
    "season", "holiday", "workingday", "weather",
    "temp", "atemp", "humidity", "windspeed",
    "hour", "weekday", "month", "year"
]

X = train[features]
y = train["count"]
X_test_final = test[features]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------
# Model Dictionary
# -----------------------------
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=20, n_jobs=-1, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=5, random_state=42
    )
}

if xgb_available:
    models["XGBoost"] = XGBRegressor(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, random_state=42
    )

if lgbm_available:
    models["LightGBM"] = LGBMRegressor(
        n_estimators=300, learning_rate=0.1, random_state=42
    )

if cat_available:
    models["CatBoost"] = CatBoostRegressor(
        iterations=300, learning_rate=0.1, depth=6,
        verbose=0, random_state=42
    )

# -----------------------------
# Model Training + MLflow Logging
# -----------------------------
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    # Fit model
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    preds[preds < 0] = 0

    # Compute metrics
    mae = mean_absolute_error(y_val, preds)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)
    rmsle = np.sqrt(mean_squared_log_error(y_val, preds))

    results[name] = {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "RMSLE": rmsle
    }

    print(f"{name} Metrics:")
    print(f"  MAE:   {mae:.2f}")
    print(f"  RMSE:  {rmse:.2f}")
    print(f"  R2:    {r2:.4f}")
    print(f"  RMSLE: {rmsle:.4f}")

    # -----------------------------
    # MLflow Run for Each Model
    # -----------------------------
    with mlflow.start_run(run_name=name):

        # Log training dataset
        training_data = pd.concat([X_train, y_train], axis=1)
        training_data_path = save_training_data(training_data)

        training_input = mlflow.data.from_pandas(
            training_data,
            targets="count",
            source=training_data_path
        )
        mlflow.log_input(training_input, context="training data")

        # Log model parameters
        mlflow.log_params(model.get_params())

        # Log metrics
        mlflow.log_metrics({
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
            "RMSLE": rmsle
        })

        # Log model artifact
        mlflow.sklearn.log_model(
            sk_model=model,
            input_example=X_val.iloc[:5],
            artifact_path="model_artifacts"
        )

# -----------------------------
# Final Model (LightGBM)
# -----------------------------
print("\n=== Model Comparison ===")
for name, metrics in results.items():
    print(f"\n{name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.4f}")

final_model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.1,
    random_state=42
)

final_model.fit(X_train, y_train)

val_preds = final_model.predict(X_val)
val_preds[val_preds < 0] = 0

rmsle = np.sqrt(mean_squared_log_error(y_val, val_preds))
print("Final LightGBM Validation RMSLE:", rmsle)

# Train on full data
final_model.fit(X, y)

test_preds = final_model.predict(X_test_final)
test_preds[test_preds < 0] = 0

submission = pd.DataFrame({
    "datetime": test["datetime"],
    "count": test_preds
})

submission.to_csv("submission_final_lightgbm.csv", index=False)
print("Saved submission_final_lightgbm.csv")

# -----------------------------------------
# Create a second CSV with features included
# -----------------------------------------
features_with_preds = X_test_final.copy()
features_with_preds["datetime"] = test["datetime"]
features_with_preds["count"] = test_preds

features_with_preds.to_csv("submission_with_features.csv", index=False)
print("Saved submission_with_features.csv")


Training Linear Regression...
Linear Regression Metrics:
  MAE:   103.50
  RMSE:  140.55
  R2:    0.4015
  RMSLE: 1.3078


C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <http


Training Random Forest...


C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <http

Random Forest Metrics:
  MAE:   24.38
  RMSE:  39.06
  R2:    0.9538
  RMSLE: 0.3277


...


Training LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000081 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 274
[LightGBM] [Info] Number of data points in the train set: 8708, number of used features: 12
[LightGBM] [Info] Start training from score 191.584750
LightGBM Metrics:
  MAE:   22.95
  RMSE:  35.82
  R2:    0.9611
  RMSLE: 0.4412


C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <http


Training CatBoost...
CatBoost Metrics:
  MAE:   24.77
  RMSE:  38.47
  R2:    0.9552
  RMSLE: 0.4989


C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
C:\Users\raherm\AppData\Local\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <http


=== Model Comparison ===

Linear Regression:
  MAE: 103.5004
  RMSE: 140.5540
  R2: 0.4015
  RMSLE: 1.3078

Random Forest:
  MAE: 24.3798
  RMSE: 39.0606
  R2: 0.9538
  RMSLE: 0.3277

Gradient Boosting:
  MAE: 24.7845
  RMSE: 39.0038
  R2: 0.9539
  RMSLE: 0.5068

XGBoost:
  MAE: 23.0634
  RMSE: 36.2971
  R2: 0.9601
  RMSLE: 0.4719

LightGBM:
  MAE: 22.9537
  RMSE: 35.8221
  R2: 0.9611
  RMSLE: 0.4412

CatBoost:
  MAE: 24.7679
  RMSE: 38.4686
  R2: 0.9552
  RMSLE: 0.4989
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000694 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 274
[LightGBM] [Info] Number of data points in the train set: 8708, number of used features: 12
[LightGBM] [Info] Start training from score 191.584750
Final LightGBM Validation RMSLE: 0.44119080238403496
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000168 seconds.
You can set `force_row_wi